JSON 데이터 병렬로 추출 전처리 작업

In [ ]:
import json
import os
from tqdm import tqdm
from multiprocess import Pool, cpu_count

In [ ]:
# 대상 폴더 내 모든 json 파일 경로 수집
json_file_paths = []
for root, _, files in os.walk(extract_path):
    for file in files:
        if file.endswith('.json'):
            json_file_paths.append(os.path.join(root, file))


In [ ]:
 각 json 파일에서 필요한 정보 추출
def parse_json(path):
    try:
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            sentence = data['Transcription']['LabelText']
            age = int(data['Speaker']['Age'])
            age_group = data['Speaker']['AgeGroup']
            return {"sentence": sentence, "age": age, "age_group": age_group}
    except:
        return None  # 잘못된 파일 예외 처리

In [ ]:
# 병렬 처리 수행
with Pool(cpu_count()) as pool:
    extracted_data = list(tqdm(pool.imap(parse_json, json_file_paths), total=len(json_file_paths)))

In [ ]:
# None 제거
extracted_data = [d for d in extracted_data if d is not None]

100%|██████████| 404375/404375 [01:52<00:00, 3578.62it/s]


(7~12세, 각 47,000개)

In [ ]:
from sklearn.utils import resample
import pandas as pd

# 원본 데이터 불러오기
df = pd.DataFrame(extracted_data)
df = df[df['age'].between(7, 12)]  # 6세 제거

# 균형 잡을 수
target_per_class = 47000
balanced_df = pd.DataFrame()

for age in range(7, 13):
    group = df[df['age'] == age]
    sampled = resample(
        group,
        replace=(len(group) < target_per_class),
        n_samples=target_per_class,
        random_state=42
    )
    balanced_df = pd.concat([balanced_df, sampled])

# 저장
balanced_df.to_csv("/content/drive/MyDrive/balanced_7to12_282k.csv", index=False)


전체 데이터 연령별 분포

In [ ]:
from tqdm import tqdm
from multiprocess import Pool, cpu_count
import pandas as pd

# DataFrame으로 변환
df = pd.DataFrame(extracted_data)

print(df['age'].value_counts().sort_index())

age
6      1500
7     61317
8     69037
9     98157
10    68434
11    56249
12    47282
Name: count, dtype: int64


In [ ]:
print(balanced_df['age'].value_counts().sort_index())

age
7     47000
8     47000
9     47000
10    47000
11    47000
12    47000
Name: count, dtype: int64


In [ ]:
!pip install --upgrade transformers
!pip install sentencepiece
!pip install gluonnlp

KoBERT 모델 및 토크나이저 불러오기

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch

tokenizer = BertTokenizer.from_pretrained("monologg/kobert")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'KoBertTokenizer'. 
The class this function is called from is 'BertTokenizer'.


데이터 전처리 (KoBERT 입력에 맞게)

In [ ]:
from sklearn.model_selection import train_test_split

# 1. 균형 잡힌 CSV 불러오기
df = pd.read_csv("/content/drive/MyDrive/balanced_7to12_282k.csv")

# 2. 나이 정수 → 클래스 인덱스로 매핑 (7세: 0, ..., 12세: 5)
df['label'] = df['age'] - 7  # 7세가 클래스 0이 되도록

# 3. 학습/검증 분리
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['sentence'].tolist(), df['label'].tolist(), test_size=0.2, random_state=42
)

# 4. KoBERT 토크나이징
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=64)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=64)


PyTorch Dataset 구성

In [ ]:
import torch

class KoBERTDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {
            'input_ids': torch.tensor(self.encodings['input_ids'][idx]),
            'attention_mask': torch.tensor(self.encodings['attention_mask'][idx]),
            'labels': torch.tensor(int(self.labels[idx]), dtype=torch.long)
        }
        return item



KoBERT 모델 불러와 학습 설정

In [ ]:
from transformers import Trainer, TrainingArguments, BertForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score

# 1. KoBERT 모델 불러오기 (클래스 수: 6)
model = BertForSequenceClassification.from_pretrained("monologg/kobert", num_labels=6)

# 2. 학습 설정
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    logging_dir='./logs',
    logging_steps=10,
    do_eval=True,
    do_train=True,
    save_steps=500
)

# 3. 성능 측정 함수 (정확도 + F1)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="macro")
    }

# 4. Trainer 정의
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)



Some weights of BertForSequenceClassification were not initialized from the model checkpoint at monologg/kobert and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


모델 학습

In [ ]:
%env CUDA_LAUNCH_BLOCKING=1
print("train_labels:", sorted(set(train_labels)))
print("val_labels:", sorted(set(val_labels)))


env: CUDA_LAUNCH_BLOCKING=1
train_labels: [0, 1, 2, 3, 4, 5]
val_labels: [0, 1, 2, 3, 4, 5]


In [ ]:
trainer.train()


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
eval_results = trainer.evaluate()
print(eval_results)


{'eval_loss': 1.781723976135254, 'eval_runtime': 41.7179, 'eval_samples_per_second': 1927.133, 'eval_steps_per_second': 30.131, 'epoch': 3.0}


In [ ]:
model.save_pretrained("/content/kobert_age_model")
tokenizer.save_pretrained("/content/kobert_age_model")


('/content/kobert_age_model/tokenizer_config.json',
 '/content/kobert_age_model/special_tokens_map.json',
 '/content/kobert_age_model/vocab.txt',
 '/content/kobert_age_model/added_tokens.json')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp -r /content/kobert_age_model /content/drive/MyDrive/kobert_age_model_backup


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
